# BÀI TẬP: E-COMMERCE DATA (ONLINE RETAIL)
**Nguồn:** kaggle.com/datasets/carrie1/ecommerce-data (541,909 dòng)


## Setup

In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_style('whitegrid')

csv_path = 'https://raw.githubusercontent.com/databricks/Spark-The-Definitive-Guide/master/data/retail-data/all/online-retail-dataset.csv'

df = pd.read_csv(csv_path, encoding='ISO-8859-1', on_bad_lines='skip')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print('Loaded from:', csv_path)
df.head()

Loaded from: https://raw.githubusercontent.com/databricks/Spark-The-Definitive-Guide/master/data/retail-data/all/online-retail-dataset.csv


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


---
# PHẦN A — DATA PROFILING
## A.1. Data size, column names, data types

In [2]:
# TODO
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  str           
 1   StockCode    541909 non-null  str           
 2   Description  540455 non-null  str           
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), str(4)
memory usage: 33.1 MB


## A.2. Missing values & Duplicate data

In [3]:
# TODO
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [5]:
df.duplicated().sum()

np.int64(5268)

## A.3. Invalid values

In [9]:
# TODO
invalid_unitprice = (df['UnitPrice'] <= 0).sum()
invalid_quantity = (df['Quantity'] <= 0).sum()

print(invalid_quantity)
print(invalid_unitprice)



10624
2517


,Quantity,UnitPrice
count,541909.000000,541909.000000
mean,9.552250,4.611114
std,218.081158,96.759853
min,-80995.000000,-11062.060000
25%,1.000000,1.250000
50%,3.000000,2.080000
75%,10.000000,4.130000
max,80995.000000,38970.000000


## A.4. Create a new column
Làm sạch dữ liệu (loại Quantity<=0, UnitPrice<=0), tạo cột `Sales` = Quantity * UnitPrice.

In [11]:
# TODO
df_clean = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)].copy()

df_clean['Sales'] = df_clean['Quantity'] * df_clean['UnitPrice']

print(df_clean.shape[0])

df_clean[['InvoiceNo', 'StockCode', 'Quantity', 'UnitPrice', 'Sales']].head()

530104


,InvoiceNo,StockCode,Quantity,UnitPrice,Sales
0,536365,85123A,6,2.55,15.30
1,536365,71053,6,3.39,20.34
2,536365,84406B,8,2.75,22.00
3,536365,84029G,6,3.39,20.34
4,536365,84029E,6,3.39,20.34


---
# PHẦN B — DESCRIPTIVE STATISTICS
## Group 1 — Central Tendency

In [15]:
# TODO
num_cols = ['Quantity', 'UnitPrice', 'Sales']

central_tendency = pd.DataFrame({
        'Mean': df_clean[num_cols].mean(),
        'Median': df_clean[num_cols].median(),
        'Mode': df_clean[num_cols].mode().iloc[0]
})

print(central_tendency.round(2))

            Mean  Median   Mode
Quantity   10.54    3.00   1.00
UnitPrice   3.91    2.08   1.25
Sales      20.12    9.90  15.00


## Group 2 — Dispersion

In [16]:
# TODO
num_cols = ['Quantity', 'UnitPrice', 'Sales']

sub = df_clean[num_cols]

q1 = sub.quantile(0.25)
q3 = sub.quantile(0.75)

dispersion_df = pd.DataFrame({
        'Range': sub.max() - sub.min(),
        'Std': sub.std(),
        'Variance': sub.var(),
        'IQR': q3 - q1
})

print(dispersion_df.round(2))

               Range     Std  Variance    IQR
Quantity    80994.00  155.52  24187.75   9.00
UnitPrice   13541.33   35.92   1289.94   2.88
Sales      168469.60  270.36  73092.77  13.95


## Group 3 — Location and Shape

In [22]:
# TODO
shape_df = pd.DataFrame({
        'Skewness': df_clean[num_cols].skew(),
        'Kurtosis': df_clean[num_cols].kurtosis(),
        'Q1 (25%)': df_clean[num_cols].quantile(0.25),
        'Q3 (75%)': df_clean[num_cols].quantile(0.75),
        'Median': df_clean[num_cols].median(),
        'P95 (95%)': df_clean[num_cols].quantile(0.95)
})

print(shape_df.round(2))

           Skewness   Kurtosis  Q1 (25%)  Q3 (75%)  Median  P95 (95%)
Quantity     471.73  236462.34      1.00     10.00    3.00      30.00
UnitPrice    206.09   62483.14      1.25      4.13    2.08       9.95
Sales        506.71  297651.66      3.75     17.70    9.90      59.70


---
# PHẦN C — DEFINE THE QUESTION

## Câu hỏi 1: Quốc gia nào đóng góp doanh thu cao nhất, chiếm bao nhiêu % tổng doanh thu?

In [24]:
# TODO
country_sales = df_clean.groupby('Country')['Sales'].sum().sort_values(ascending=False)
total_sales = country_sales.sum()
country_pct = (country_sales/total_sales) * 100

country_df = pd.DataFrame({
    'Total_sales': country_sales,
    'Sales_share(%)': country_pct
})

print(country_df.round(2))

                      Total_sales  Sales_share(%)
Country                                          
United Kingdom         9025222.08           84.61
Netherlands             285446.34            2.68
EIRE                    283453.96            2.66
Germany                 228867.14            2.15
France                  209715.11            1.97
Australia               138521.31            1.30
Spain                    61577.11            0.58
Switzerland              57089.90            0.54
Belgium                  41196.34            0.39
Sweden                   38378.33            0.36
Japan                    37416.37            0.35
Norway                   36165.44            0.34
Portugal                 33747.10            0.32
Finland                  22546.08            0.21
Singapore                21279.29            0.20
Channel Islands          20450.44            0.19
Denmark                  18955.34            0.18
Italy                    17483.24            0.16


## Câu hỏi 2: Sản phẩm nào bán chạy nhất theo doanh thu?

In [25]:
# TODO
top_products = df_clean.groupby(['StockCode', 'Description'])['Sales'].agg(['sum', 'count']).reset_index()
top_products = top_products.rename(columns={'sum': 'Total_Revenue', 'count': 'Transaction_Count'})
top_products = top_products.sort_values(by='Total_Revenue', ascending=False)

print(top_products.head(10).round(2))

best_product = top_products.iloc[0]
print(best_product)

     StockCode                         Description  Total_Revenue  \
4150       DOT                      DOTCOM POSTAGE      206248.77   
1340     22423            REGENCY CAKESTAND 3 TIER      174484.74   
2668     23843         PAPER CRAFT , LITTLE BIRDIE      168469.60   
3640    85123A  WHITE HANGING HEART T-LIGHT HOLDER      104340.29   
2877     47566                       PARTY BUNTING       99504.33   
3619    85099B             JUMBO BAG RED RETROSPOT       94340.05   
2123     23166      MEDIUM CERAMIC TOP STORAGE JAR       81700.92   
4151         M                              Manual       78110.27   
4153      POST                             POSTAGE       78101.88   
2029     23084                  RABBIT NIGHT LIGHT       66964.99   

      Transaction_Count  
4150                706  
1340               2017  
2668                  1  
3640               2256  
2877               1706  
3619               2112  
2123                250  
4151                321  
4153  

## Câu hỏi 3: Doanh số có tính mùa vụ theo tháng không?

In [26]:
# TODO
df_clean['YearMonth'] = df_clean['InvoiceDate'].dt.to_period('M')

monthly_sales = df_clean.groupby('YearMonth')['Sales'].agg(Total_Sales='sum', Total_Orders='nunique')
print(monthly_sales.round(2))


           Total_Sales  Total_Orders
YearMonth                           
2010-12      823746.14          1448
2011-01      691364.56          1404
2011-02      523631.89          1100
2011-03      717639.36          1339
2011-04      537808.62          1158
2011-05      770536.02          1295
2011-06      761739.90          1434
2011-07      719221.19          1371
2011-08      759138.38          1374
2011-09     1058590.17          1496
2011-10     1154979.30          1692
2011-11     1509496.33          2026
2011-12      638792.68          1296


## Câu hỏi 4: Giá trị đơn hàng trung bình (Average Order Value) khác nhau thế nào giữa các quốc gia?

In [27]:
# TODO

order_sales = df_clean.groupby(['Country', 'InvoiceNo'])['Sales'].sum().reset_index()

aov_country = order_sales.groupby('Country')['Sales'].agg(AOV='mean', Median_Order_Value='median', Order_Count='count').sort_values(by='AOV', ascending=False)

print(aov_country.round(2))

                          AOV  Median_Order_Value  Order_Count
Country                                                       
Singapore             3039.90             2118.74            7
Netherlands           3036.66              806.88           94
Australia             2430.20              429.60           57
Japan                 1969.28             1607.04           19
Lebanon               1693.88             1693.88            1
Hong Kong             1426.53             1539.64           11
Brazil                1143.60             1143.60            1
Sweden                1066.06              372.18           36
Switzerland           1057.22              625.96           54
Denmark               1053.07              515.10           18
Israel                1016.91              401.78            8
Norway                1004.60              631.27           36
RSA                   1002.31             1002.31            1
EIRE                   984.22              651.72      

## Câu hỏi 5: Tỷ lệ giao dịch có dấu hiệu trả hàng/hủy (Quantity âm ở dữ liệu gốc) khác nhau thế nào giữa các quốc gia?

In [28]:
# TODO
df['is_cancelled'] = df['Quantity'] < 0

cancel_stats = df.groupby('Country')['is_cancelled'].agg(Total_Txns='count', Cancelled_Txns='sum', Cancellation_Rate='mean')
cancel_stats['Cancellation_Rate (%)'] = cancel_stats['Cancellation_Rate'] * 100

print(cancel_stats[cancel_stats['Total_Txns'] >= 100].sort_values(by='Total_Txns', ascending=False).head(10).round(2))

                Total_Txns  Cancelled_Txns  Cancellation_Rate  \
Country                                                         
United Kingdom      495478            9192               0.02   
Germany               9495             453               0.05   
France                8557             149               0.02   
EIRE                  8196             302               0.04   
Spain                 2533              48               0.02   
Netherlands           2371               8               0.00   
Belgium               2069              38               0.02   
Switzerland           2002              35               0.02   
Portugal              1519              18               0.01   
Australia             1259              74               0.06   

                Cancellation_Rate (%)  
Country                                
United Kingdom                   1.86  
Germany                          4.77  
France                           1.74  
EIRE               

## Câu hỏi 6 (Tổng hợp) — Viết insight tổng hợp
Dựa trên Phần A, B, C, viết 4-5 câu insight tổng thể.

*(Viết insight của bạn vào đây...)*